`.rst` means **reStructuredText**.

It is a plain-text documentation format, similar in purpose to Markdown (`.md`), but more common in Python/Sphinx documentation.

In this repo, files like:

```bash
doc_source/source/tutorials/sxrd.rst
```

are source files for the GenX documentation. Sphinx reads these `.rst` files and generates HTML pages like:

```bash
docs/doc/tutorials/sxrd.html
```

Some common `.rst` syntax:

```rst
Section Title
=============

Subsection
----------

This is normal text.

* bullet item
* another bullet item

Code block::

    import models.sxrd as model
    print("hello")

Link to another page:

:ref:`tutorial-sxrd`

Download link:

:download:`rods.dat <_attachments/surfdiff/rods.dat>`

Menu instruction:

:menuselection:`Settings-->Data Loaders`
```

So for your file, `sxrd.rst` is not Python code itself. It is a documentation/tutorial file that contains explanatory text plus embedded code examples.

---

That code is from the SXRD tutorial, here:

- Tutorial source: [sxrd.rst](/data/users/chuanyul/genx/doc_source/source/tutorials/sxrd.rst:49)
- Generated docs copy: [sxrd.html](/data/users/chuanyul/genx/docs/doc/tutorials/sxrd.html)
- Actual SXRD model implementation: [sxrd.py](/data/users/chuanyul/genx/genx/genx/models/sxrd.py)
- Symmetry helper implementation: [symmetries.py](/data/users/chuanyul/genx/genx/genx/models/symmetries.py)
- SXRD data loader: [sls_sxrd.py](/data/users/chuanyul/genx/genx/genx/plugins/data_loaders/sls_sxrd.py)
- Rod-data generator you opened: [make_sxrd_data.py](/data/users/chuanyul/genx/doc_source/source/tutorials/_attachments/surfdiff/make_sxrd_data.py)

There is **not** a separate `.py` file containing exactly that full model script. It is embedded in the tutorial file [sxrd.rst](/data/users/chuanyul/genx/doc_source/source/tutorials/sxrd.rst:49). You would usually paste/write this inside the GenX model script editor.

What the code does:

```python
import models.sxrd as model
```

Inside GenX, this imports the SXRD model module. In normal standalone Python, the package path would more likely be:

```python
from genx.models import sxrd as model
```

---

**The whole model means: simulate SXRD rod intensities for a LaAlO3 film on SrTiO3 substrate using p4 surface symmetry.**


```python
unitcell = model.UnitCell(3.945, 3.945, 3.945, 90, 90, 90)
```

This defines the substrate unit cell. Here it is cubic with:

```text
a = b = c = 3.945 Angstrom
alpha = beta = gamma = 90 degrees
```

This is used to convert reciprocal coordinates `(h, k, l)` into physical scattering geometry.

```python
inst = model.Instrument(wavel=1.0, alpha=1.0)
```

This defines the X-ray instrument: wavelength `1.0 A`, incidence angle `1.0 degree`.

```python
bulk = model.Slab()
```

This creates the bulk substrate slab. The next lines add SrTiO3 atoms:

```python
bulk.add_atom('Sr', 'sr', 0.0, 0.0, 0.0, 0.08, 1.0)
bulk.add_atom('Ti', 'ti', 0.5, 0.5, 0.5, 0.08, 1.0)
...
```

The pattern is:

```python
add_atom(id, element, x, y, z, u, occupancy)
```

Where:

- `id`: unique atom name inside the slab
- `element`: scattering element key, like `'sr'`, `'ti'`, `'o'`
- `x, y, z`: fractional coordinates
- `u`: Debye-Waller/displacement parameter
- `occupancy`: site occupancy

Then:

```python
laouc = model.Slab(c=1.05)
```

This creates a LaAlO3 film slab. `c=1.05` means the slab’s out-of-plane `c` direction is scaled 5% longer than the substrate unit cell.

```python
laouc.add_atom('La', 'la', 0.0, 0.0, 0.0, 0.08, 1.0, 1)
```

For film atoms, there is one extra argument:

```python
add_atom(id, element, x, y, z, u, occupancy, multiplicity)
```

The final number is the site multiplicity under the surface symmetry.

This part defines fourfold rotational symmetry:

```python
p4 = [
    model.SymTrans([[1, 0], [0, 1]]),
    model.SymTrans([[-1, 0], [0, -1]]),
    model.SymTrans([[0, -1], [1, 0]]),
    model.SymTrans([[0, 1], [-1, 0]])
]
```

These are 2D matrix transformations on `(x, y)` coordinates: identity, 180-degree rotation, +90-degree rotation, and -90-degree rotation.

Then:

```python
sample = model.Sample(inst, bulk, [laouc], unitcell)
sample.set_surface_sym(p4)
```

This combines everything into one sample: instrument, bulk substrate, one film slab, unit cell, and surface symmetry.

Finally, this is the function GenX calls during simulation:

---
```python
def Sim(data):
    I = []
    for data_set in data:
        h = data_set.extra_data['h']
        k = data_set.extra_data['k']
        l = data_set.x
        f = sample.calc_f(h, k, l)
        i = abs(f)**2
        I.append(i)
    return I
```
---


Each `data_set` is one SXRD rod, for example `(h, k) = (1, 0)`. The loader stores `h` and `k` in:

```python
data_set.extra_data['h']
data_set.extra_data['k']
```

and stores the varying `l` values in:

```python
data_set.x
```

**Then calculate the complex structure factor**:

```python
f = sample.calc_f(h, k, l)
```

**Converts it to intensity**

```python
i = abs(f)**2
```

---

## `calc_f` is the heart of the model

```python
f = sample.calc_f(h, k, l)
```

- `sample` knows everything: the bulk slab, the surface slab(s), the unit cell, the symmetry, the instrument.
- `calc_f` walks through that whole structure and returns the **complex structure factor F(h, k, l)** at every requested reciprocal-space point.
- Everything else in `Sim` is bookkeeping; this single call *is* the physics.

**What is the structure factor?**

The **structure factor F(h, k, l)** is the Fourier transform of the electron density of the unit cell, evaluated at the reciprocal-lattice point (h, k, l):

$$F(h,k,l) = \sum_{j} f_j(Q)\, e^{2\pi i (h x_j + k y_j + l z_j)}\, e^{-2\pi^2 u_j Q^2}\, \text{occ}_j$$

The pieces map directly onto what you wrote in `add_atom`:

| Symbol | Meaning | Where it came from in the script |
|---|---|---|
| $f_j(Q)$ | atomic form factor of atom *j* (element-dependent) | the `element` argument |
| $(x_j, y_j, z_j)$ | fractional position | the `x, y, z` args + any refined `dx, dy, dz` |
| $u_j$ | Debye-Waller factor (thermal motion) | the `u` arg |
| $\text{occ}_j$ | site occupancy | the `occ` arg |
| sum over *j* | every atom in the unit cell | all the `add_atom` calls (with `mult` expanding for symmetry) |

So `calc_f` is just doing this sum over every atom in the bulk + surface stack, with the right phase factors for each (h, k, l) you ask about.

**Why we care: F is the bridge between model and measurement**

In an X-ray diffraction experiment, the detector counts photons. The number of photons scattered into a given (h, k, l) is proportional to **|F(h, k, l)|²** (times some geometric and instrumental factors).

So the workflow is:

```
your atomic model  ──calc_f──▶  F(h,k,l)  ──|·|²──▶  predicted I(h,k,l)
                                                          │
                                                          ▼
                                                  compared to measured I
                                                          │
                                                          ▼
                                              fit adjusts atom positions
```

That's why `Sim` ends with `i = abs(f)**2` — the experiment can't see F directly (it loses the phase, the famous **phase problem**), only |F|². The fitting algorithm (differential evolution) adjusts atomic parameters until |F_calc|² matches |F_meas|².

**Why F is special for SXRD specifically**

In bulk crystallography, F is just sampled at integer (h, k, l). In SXRD it's sampled **continuously along l** at integer (or fractional) (h, k) — that's the CTR.

The CTR shape comes from **interference** in F:

- **At Bragg points (integer l):** every bulk layer contributes the same phase. F adds up constructively → giant intensity.
- **At anti-Bragg (half-integer l):** successive bulk layers alternate sign and cancel pairwise. The bulk part of F nearly vanishes, and the surface atoms — unpaired by any partner below — **dominate F**.

In equation form, with N bulk unit cells:

$$F_{\text{total}} = F_{\text{surface}} + F_{\text{bulk-unit}} \cdot \sum_{n=0}^{\infty} e^{2\pi i l n} e^{-\beta n}$$

That geometric sum (the *truncation rod factor*) goes like 1/(1 − e^{2πil}) — which is **huge near integer l** and **small near half-integer l**. So F_surface, which is a small number on its own, becomes a *relatively* large fraction of F_total exactly at anti-Bragg. That's why anti-Bragg is the surface-sensitive regime.

**Why it has to be complex, not just a magnitude**

Two reasons:

1. **Interference between bulk and surface.** F_bulk and F_surface add as **complex amplitudes**, not magnitudes. A surface atom shifted upward by 0.1 Å rotates its phase contribution and can either enhance or cancel the bulk truncation tail, depending on l. You only get this right by tracking the full complex F.
2. **Stacking multiple slabs.** With a 2-unit-cell LAO film on STO, F_film is itself a sum of complex contributions from each LAO layer with phase factor e^{2πilz}. Squaring magnitudes too early would lose all that interference.

So `calc_f` returns a complex array, and `abs(f)**2` is taken **once at the very end**.

**What sensitivity does this give you?**

Because F encodes phase as well as magnitude, and because the CTR samples F continuously along l, |F(l)|² is sensitive to:

- **Atomic z-positions** in the surface layer — they shift the phase of F_surface relative to F_bulk, changing the asymmetry of the CTR around each Bragg peak.
- **Occupancies and roughness** — they reduce the magnitude of F_surface, damping the CTR modulation between Bragg peaks.
- **Reconstructions** — they appear as F_surface contributions at fractional (h, k), where F_bulk = 0. So superstructure rod intensities are *purely* |F_surface|².
- **Debye-Waller factors** — they damp F at high Q (high l), tilting the overall envelope.

All of this is buried inside that one `calc_f` call.

- `sample.calc_f(h, k, l)` evaluates the complex structure factor — the Fourier transform of your modeled electron density — at every reciprocal-space point you ask about.
- The experiment measures |F|², so this is **the** quantity that connects your atomic model to the data.
- For SXRD specifically, the *interference* between F_bulk and F_surface along l is what creates the CTR shape, and what makes the technique sensitive to surface atoms in the first place.
- That's why every refinable surface parameter — z-positions, occupancies, Debye-Waller factors — has to flow through `calc_f` to be testable against the data.